# Model comparison

Load one or more **trained** ShearNet models, evaluate each on a freshly
simulated test set matched to how it was trained, and compare them: learning
curves, an MSE / bias table, prediction scatter and residual distributions —
optionally against an **NGmix metacalibration** baseline.

This notebook reuses the exact functions behind the `shearnet-eval` CLI
(`generate_test_data`, `load_model`, `run_shearnet`), so results match the CLI.

**Prerequisites** — one or more models trained with `shearnet-train` (or
`01_quickstart.ipynb` with `save_path` set). Each model is expected at:

| Artifact | Location |
|---|---|
| checkpoint | `$SHEARNET_DATA_PATH/model_checkpoint/<name>...` |
| training config | `$SHEARNET_DATA_PATH/plots/<name>/training_config.yaml` |
| loss history | `$SHEARNET_DATA_PATH/plots/<name>/<name>_loss.npz` |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from shearnet.config.config_handler import Config
from shearnet.cli.evaluate import generate_test_data, load_model, run_shearnet
from shearnet.metrics import eval_ngmix

DATA_PATH = os.getenv("SHEARNET_DATA_PATH", os.path.abspath("."))
PLOT_PATH = os.path.join(DATA_PATH, "plots")
SAVE_PATH = os.path.join(DATA_PATH, "model_checkpoint")
print("Reading models from:", DATA_PATH)

## 1 · Choose the models to compare

Put the names of the models you trained in `MODELS`. `TEST_SAMPLES` / `SEED`
override each model's saved evaluation settings so all models see a test set of
the same size.

In [ ]:
MODELS = [
    "quickstart_demo",
    # "fork-like_low_noise",
    # "research_backed_high_noise",
]

TEST_SAMPLES = 2000
SEED = 58
RUN_NGMIX = False   # also run NGmix metacalibration on the first model's test set (slower)

COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]

## 2 · Rebuild each model's evaluation config

`build_eval_config` reconstructs the flat settings dict that `shearnet-eval`
assembles internally, reading each model's saved `training_config.yaml` so the
simulated test set matches how the model was trained.

In [ ]:
def build_eval_config(model_name, test_samples=TEST_SAMPLES, seed=SEED):
    cfg_path = os.path.join(PLOT_PATH, model_name, "training_config.yaml")
    config = Config(cfg_path) if os.path.exists(cfg_path) else Config()
    return {
        "model_name": model_name,
        "seed": seed if seed is not None else config.get("dataset.seed", 58),
        "test_samples": test_samples,
        "psf_fwhm": config.get("dataset.psf_fwhm", 0.25),
        "nse_sd": config.get("dataset.nse_sd", 1e-5),
        "exp": config.get("dataset.exp", "ideal"),
        "stamp_size": config.get("dataset.stamp_size", 53),
        "pixel_size": config.get("dataset.pixel_size", 0.141),
        "apply_psf_shear": config.get("dataset.apply_psf_shear", False),
        "psf_shear_range": config.get("dataset.psf_shear_range", 0.05),
        "process_psf": config.get("model.process_psf", False),
        "nn": config.get("model.type", "cnn"),
        "galaxy_type": config.get("model.galaxy.type", "research_backed"),
        "psf_type": config.get("model.psf.type", "forklens_psf"),
        "fusion": config.get("model.fusion", "concat"),
        "output_keys": tuple(config.get("model.output_keys", ("g1", "g2"))),
        "gap": config.get("model.gap", False),
        "psf_model": config.get("comparison.psf_model", "gauss"),
        "gal_model": config.get("comparison.gal_model", config.get("dataset.type", "gauss")),
        "cosmos_cat_fname": config.get("catalog.cosmos_cat_fname"),
        "psfex_model_file": config.get("dataset.psfex_model_file"),
        "hlr_type": config.get("dataset.hlr_type", "constant"),
        "flux_type": config.get("dataset.flux_type", "constant"),
        "save_path": SAVE_PATH,
        "plot_path": PLOT_PATH,
        "mcal": False,
        "plot": False,
    }

## 3 · Learning curves

Read directly from the saved `<name>_loss.npz` files — no model loading needed.

In [ ]:
plt.figure(figsize=(11, 7))
for name, color in zip(MODELS, COLORS):
    loss_file = os.path.join(PLOT_PATH, name, f"{name}_loss.npz")
    if not os.path.exists(loss_file):
        print(f"[skip] no loss file for {name}: {loss_file}")
        continue
    d = np.load(loss_file, allow_pickle=True)
    tl, vl = d["train_loss"], d["val_loss"]
    plt.plot(np.arange(1, len(tl) + 1), tl, color=color, ls="-", alpha=0.6,
             label=f"{name} — train")
    plt.plot(np.arange(1, len(vl) + 1), vl, color=color, ls="--", lw=2,
             label=f"{name} — val")
    best = int(np.argmin(vl))
    print(f"{name}: best val {float(vl[best]):.3e} @ epoch {best + 1} "
          f"({len(tl)} epochs total)")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Learning curves")
plt.legend(fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4 · Evaluate each model

For every model with a checkpoint we simulate its matched test set, restore the
network, and predict. Missing models are skipped with a message rather than
raising.

In [ ]:
results = {}   # name -> dict(preds, labels, obs, mse, bias, time, cfg)
for name in MODELS:
    ckpts = ([d for d in os.listdir(SAVE_PATH) if d.startswith(name)]
             if os.path.isdir(SAVE_PATH) else [])
    if not ckpts:
        print(f"[skip] no checkpoint for '{name}' in {SAVE_PATH}")
        continue
    print(f"\n=== {name} ===")
    cfg = build_eval_config(name)
    gal, psf, labels, obs = generate_test_data(cfg)
    state = load_model(cfg, gal, psf)
    preds, mse, bias, elapsed = run_shearnet(state, gal, psf, labels, cfg)
    results[name] = dict(preds=np.asarray(preds), labels=np.asarray(labels),
                         obs=obs, mse=mse, bias=bias, time=elapsed, cfg=cfg)

print(f"\nEvaluated {len(results)} model(s).")

In [ ]:
# Summary table.
if results:
    print(f"{'model':<30}{'MSE (g1,g2)':>14}{'bias':>14}{'time [s]':>10}")
    print("-" * 68)
    for name, r in results.items():
        print(f"{name:<30}{r['mse']:>14.3e}{r['bias']:>+14.3e}{r['time']:>10.2f}")
else:
    print("No models evaluated - set MODELS to names you have trained.")

## 5 · Prediction scatter & residuals

Shear components `g1`, `g2` are the first two outputs of every ShearNet model, so
these panels work regardless of how many extra parameters a model predicts.

In [ ]:
PLOT_KEYS = {"g1": 0, "g2": 1}

if results:
    fig, axes = plt.subplots(1, len(PLOT_KEYS), figsize=(6 * len(PLOT_KEYS), 6))
    axes = np.atleast_1d(axes)
    for ax, (key, j) in zip(axes, PLOT_KEYS.items()):
        for (name, r), color in zip(results.items(), COLORS):
            ax.scatter(r["labels"][:, j], r["preds"][:, j], s=6, alpha=0.35,
                       color=color, label=name)
        ax.plot([-1, 1], [-1, 1], "r--", label="y = x")
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(f"{key} true")
        ax.set_ylabel(f"{key} predicted")
        ax.set_title(key)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if results:
    fig, axes = plt.subplots(1, len(PLOT_KEYS), figsize=(6 * len(PLOT_KEYS), 4.5))
    axes = np.atleast_1d(axes)
    for ax, (key, j) in zip(axes, PLOT_KEYS.items()):
        for (name, r), color in zip(results.items(), COLORS):
            res = r["preds"][:, j] - r["labels"][:, j]
            lo, hi = np.percentile(res, [1, 99])
            res = res[(res >= lo) & (res <= hi)]
            ax.hist(res, bins=50, histtype="step", lw=2, color=color,
                    density=True, label=name)
        ax.axvline(0, color="red", ls="--")
        ax.set_title(f"{key} residuals (pred − true)")
        ax.set_xlabel("residual")
        ax.set_ylabel("density")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6 · NGmix metacalibration baseline (optional)

Set `RUN_NGMIX = True` above to also fit the first model's test set with NGmix
metacalibration for an apples-to-apples accuracy comparison. This is slower
(seconds per hundred galaxies).

In [ ]:
if RUN_NGMIX and results:
    name0 = next(iter(results))
    r = results[name0]
    print(f"Running NGmix on the '{name0}' test set ({len(r['obs'])} galaxies)...")
    ng = eval_ngmix(
        r["obs"], r["labels"],
        seed=r["cfg"]["seed"],
        psf_model=r["cfg"]["psf_model"],
        gal_model=r["cfg"]["gal_model"],
    )
    print(f"\nShearNet ({name0}) MSE(g1,g2): {r['mse']:.3e}")
    print(f"NGmix                MSE(g1,g2): {float(ng['loss']):.3e}")
elif not RUN_NGMIX:
    print("RUN_NGMIX is False - skipping the NGmix baseline.")